In [7]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, StringType
from pyspark.sql import functions as F
from pathlib import Path

In [8]:

spark = SparkSession.builder \
        .master('local[*]')\
        .appName('ValidatorTransactionSpark') \
        .getOrCreate()

ruta = Path.cwd().parents[0] / 'data'/'transacciones_practica.csv'



spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)





In [9]:
def cargar_datos(spark, ruta_csv):
    
    """ representa la estructura completa de la tabla.
        Cada StructField(...) define una columna"""
    schema = StructType([
        StructField('id', LongType(), True), # el 3er @arg indica que la columna puede contener valores nulos. 
        StructField('cliente_id', LongType(), True),
        StructField('valor', StringType(), True),
    ])

    df = (
        spark.read
        .option('header', True) # indica a Spark que la primera fila del CSV contiene los nombres de las columnas
        .schema(schema)
        .csv(str(ruta_csv))
    )

    return df

In [10]:
df_raw = cargar_datos(spark,ruta)

df_raw

id,cliente_id,valor
1,101,778669
2,102,664511
3,103,221200
4,104,340659
5,105,390826
6,106,525002
7,107,622061
8,108,23207
9,109,241119
10,110,408150


In [11]:
# regexp_replace(string, pattern, replacement)
def limpiar_col(df: DataFrame):
    df = df.withColumn(
        "valor",
        F.trim(F.regexp_replace(F.col("valor"), "[$,]", "").cast(IntegerType()))
    )
    return df

limpiar_col(df_raw)

id,cliente_id,valor
1,101,778669
2,102,664511
3,103,221200
4,104,340659
5,105,390826
6,106,525002
7,107,622061
8,108,23207
9,109,241119
10,110,408150


In [ ]:
def obtener_valores_incorrectos_col_valor(df_limpiados:DataFrame):
    return df_limpiados.filter((F.col('valor').isNull) | (F.col('valor') <= 0) | (F.col('valor').isNaN))

obtener_valores_incorrectos_col_valor( limpiar_col(df_raw))

AttributeError: 'function' object has no attribute '_get_object_id'